# Notebook Mentoría: Proyecto Final

**Fuentes de datos:**
- `backup_tracks.csv` — tracks enriquecidos via `track.getInfo` (nombre, artista, duración en ms, tag, playcount, listeners, published)
- `lastfm_dataset.csv` — tracks del pipeline multi-endpoint (aporta: country, genre_tag, rank_global, rank_by_country)

**Dataset de trabajo:** `df_merged` = merge de ambas fuentes por `mbid`

---

## Variables del proyecto

| Nombre | Significado |
| --- | --- |
| `df_raw` | `backup_tracks.csv` cargado directamente |
| `df_lastfm` | `lastfm_dataset.csv` cargado directamente |
| `df_merged` | Resultado del merge por `mbid` |
| `df_clean` | `df_merged` limpio con tipos correctos |
| `df_analysis` | `df_clean` con todas las features creadas |
| `df_tags` | DataFrame de tags con name, count, reach |
| `df_tracks` | Tracks recogidos via API por tag |

## Pipeline del notebook

1. Importaciones y configuración
2. Configuración API
3. Extracción de datos (tags + tracks via API)
4. Construcción de DataFrames (`df_tags`, `df_tracks`)
5. Merge y construcción de `df_merged`
6. Limpieza de datos
7. Feature Engineering
8. EDA
9. Correlaciones
10. Tests estadísticos
11. Modelado
12. Conclusiones

---
## 1. Importaciones y configuración

In [2]:
import os
import ast
import time
import warnings
import pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
# import statsmodels.api as sm

from scipy.stats import spearmanr, mannwhitneyu, kruskal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print('✅ Imports correctos')

✅ Imports correctos


---
## 2. Configuración API

| Endpoint | Qué devuelve | Para qué lo usamos |
|---|---|---|
| `tag.getTopTags` | Tags globales con name, count, reach | Construir `df_tags` |
| `tag.getTopTracks` | Tracks por género | Construir `df_tracks` |
| `track.getInfo` | Metadata completa por mbid | Crea `backup_tracks.csv` |

> **CORRECCIÓN #1 — API sin validación:** el código original hacía `re.json()` sin comprobar `status_code`.  
> Si la API devuelve 403/429/500, `.json()` lanza excepción no controlada.  
> **Solución:** función `get_json()` centralizada con validación y `try/except`.

In [24]:
API_KEY  = '63e059c3c912a3f642daf2372484d183'
BASE_URL = f'http://ws.audioscrobbler.com/2.0/?api_key={API_KEY}&'

# CORRECCIÓN #1: función centralizada con validación de status_code y control de errores
def get_json(url):
    """Petición GET con validación de status y manejo de errores de red."""
    try:
        time.sleep(0.5)  # delay para no superar rate limit (~4 req/s, límite = 5)
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            print(f'  ⚠️  HTTP {resp.status_code} para: {url[:80]}')
            return None
        return resp.json()
    except requests.exceptions.RequestException as e:
        print(f'  ⚠️  Error de red: {e}')
        return None

print('✅ API configurada')

✅ API configurada


---
## 3. Extracción de datos

### 3.1 Get top tags → `df_tags`

> **CORRECCIÓN #2 — Tags solo guardaban `name`:** el código original solo extraía el nombre a una lista plana.  
> La API devuelve también `count` y `reach`, útiles para filtrar tags relevantes.  
> **Solución:** capturar los tres campos y construir `df_tags` como DataFrame enriquecido.

In [5]:
# Petición a tag.getTopTags con validación
re_json = get_json(f'{BASE_URL}method=tag.getTopTags&format=json')

if re_json is None:
    print('❌ No se pudo obtener los tags. Comprueba la API key o la conexión.')
else:
    print('✅ Respuesta recibida')
    print('Primer tag:', re_json.get('toptags', {}).get('tag', [{}])[0])

✅ Respuesta recibida
Primer tag: {'name': 'rock', 'count': 4069787, 'reach': 402949}


In [6]:
# CORRECCIÓN #2: df_tags con estructura enriquecida (name, count, reach)
raw_tags = re_json.get('toptags', {}).get('tag', []) if re_json else []

df_tags = pd.DataFrame([
    {
        'name' : t.get('name', ''),
        'count': int(t.get('count', np.nan)),
        'reach': int(t.get('reach', np.nan))
    }
    for t in raw_tags
])

print(f'df_tags: {df_tags.shape[0]} tags')
df_tags.head(10)

df_tags: 50 tags


,name,count,reach
0,rock,4069787,402949
1,electronic,2499390,262248
2,seen live,2194812,82562
3,alternative,2131102,267226
4,pop,2082966,233792
5,indie,2065531,260479
6,female vocalists,1634615,169166
7,metal,1305863,159028
8,alternative rock,1230552,170275
9,jazz,1196709,149996


In [7]:
# tag_list: lista de nombres (compatibilidad con el loop existente)
# Ahora se construye desde df_tags en lugar de un loop manual
tag_list = df_tags['name'].tolist()
print(f'Total tags: {len(tag_list)}')
print('Primeros 10:', tag_list[:10])

Total tags: 50
Primeros 10: ['rock', 'electronic', 'seen live', 'alternative', 'pop', 'indie', 'female vocalists', 'metal', 'alternative rock', 'jazz']


In [16]:
import os

TAGS_FILE = 'tags_dataset.csv'

if os.path.exists(TAGS_FILE):
    df_tags = pd.read_csv(TAGS_FILE)
    print('📂 Tags cargados desde CSV')
else:
    print('🚀 Descargando tags...')
    
    raw_tags = re_json.get('toptags', {}).get('tag', []) if re_json else []

    df_tags = pd.DataFrame([
        {
            'name' : t.get('name', ''),
            'count': int(t.get('count', np.nan)),
            'reach': int(t.get('reach', np.nan))
        }
        for t in raw_tags
    ])

    df_tags.to_csv(TAGS_FILE, index=False)
    print('✅ Tags guardados')

🚀 Descargando tags...
✅ Tags guardados


### 3.2 Get top tracks por tag → `df_tracks`

In [17]:
# Prueba de salida antes del loop completo
prueba_json = get_json(f'{BASE_URL}method=tag.gettoptracks&tag=rock&format=json&limit=5')

if prueba_json:
    muestra = prueba_json.get('tracks', {}).get('track', [])
    print(f'Estructura OK: {len(muestra)} tracks de prueba')
    print('Campos:', list(muestra[0].keys()) if muestra else 'vacío')

Estructura OK: 5 tracks de prueba
Campos: ['name', 'duration', 'mbid', 'url', 'streamable', 'artist', 'image', '@attr']


In [18]:
# Loop sobre tag_list para recoger tracks
# CORRECCIÓN #1 aplicada: get_json() valida status_code y tiene try/except

tracks = []

for tag in tag_list:
    data_json = get_json(f'{BASE_URL}method=tag.gettoptracks&tag={tag}&format=json&limit=1000')

    if not data_json:  # si falló la petición, saltamos este tag
        continue

    tag_tracks = data_json.get('tracks', {}).get('track', [])

    for t in tag_tracks:
        artist_name = t.get('artist', {}).get('name', '') if isinstance(t.get('artist'), dict) else ''
        tracks.append({
            'name'  : t.get('name', ''),
            'artist': artist_name,
            'mbid'  : t.get('mbid', np.nan)
        })

print(f'Total tracks recogidos: {len(tracks):,}')

Total tracks recogidos: 48,134


---
## 4. Construcción de `df_tracks`

In [19]:
df_tracks = pd.DataFrame(tracks)
df_tracks.info()
df_tracks.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 48134 entries, 0 to 48133
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    48134 non-null  str  
 1   artist  48134 non-null  str  
 2   mbid    46974 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB


,name,artist,mbid
0,The Chain - 2004 Remaster,Fleetwood Mac,cd06c484-9319-4376-a104-504871e19756
1,Iris,Goo Goo Dolls,02a75d1a-977b-4430-9db4-5d02d26e1d85
2,Everlong,Foo Fighters,00779e5b-e581-3fcb-b0af-d6150e446b23


In [20]:
print(f'Duplicados totales: {df_tracks.duplicated().sum():,}')
print(f'Duplicados con mbid NaN: {df_tracks[df_tracks.duplicated()]["mbid"].isna().sum():,}')

Duplicados totales: 12,751
Duplicados con mbid NaN: 194


In [21]:
# CORRECCIÓN #3 — dropna() global destructivo:
# El original hacía df_tracks.dropna(inplace=True), eliminando filas por NaN en cualquier columna.
# IMPACTO: pérdida masiva de datos. Solo mbid es necesario para track.getInfo.
# SOLUCIÓN: limitar dropna() solo a columnas críticas.

df_tracks.drop_duplicates(inplace=True)
df_tracks.dropna(subset=['mbid'], inplace=True)  # solo eliminamos sin mbid
df_tracks.info()

<class 'pandas.DataFrame'>
Index: 34417 entries, 0 to 48133
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    34417 non-null  str  
 1   artist  34417 non-null  str  
 2   mbid    34417 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB


In [14]:
# Prueba de track.getInfo con un mbid real
prueba = get_json(f'{BASE_URL}method=track.getInfo&mbid={df_tracks["mbid"].iloc[0]}&format=json')
print('Campos de track:', list(prueba.get('track', {}).keys()) if prueba else 'error')

Campos de track: ['name', 'mbid', 'url', 'duration', 'streamable', 'listeners', 'playcount', 'artist', 'album', 'toptags', 'wiki']


In [15]:
# Enriquecimiento con track.getInfo → genera backup_tracks.csv
# CORRECCIÓN #1 aplicada: get_json() con validación de status_code

tracks_info = []

for mbid in df_tracks['mbid']:
    data_json = get_json(f'{BASE_URL}method=track.getInfo&mbid={mbid}&format=json')

    if not data_json:
        continue

    t = data_json.get('track')
    if t:
        artist_name = t.get('artist', {}).get('name', np.nan) if isinstance(t.get('artist'), dict) else np.nan
        tracks_info.append({
            'name'      : t.get('name', np.nan),
            'artist'    : artist_name,
            'duration'  : t.get('duration', np.nan),  # milisegundos
            'mbid'      : t.get('mbid', np.nan),
            'tag'       : t.get('toptags', {}).get('tag', []),
            'listeners' : t.get('listeners', np.nan),
            'playcount' : t.get('playcount', np.nan),
            'published' : t.get('wiki', {}).get('published', np.nan)
        })

print(f'Tracks enriquecidos: {len(tracks_info):,}')

# Guardar como backup_tracks.csv
pd.DataFrame(tracks_info).to_csv('backup_tracks.csv', index=False)
print('✅ backup_tracks.csv guardado')

KeyboardInterrupt: 

---
## 5. Merge: `backup_tracks` + `lastfm_dataset` → `df_merged`

**Por qué hacemos este merge:**
- `backup_tracks.csv` tiene: `duration` (ms), `tag`, `listeners`, `playcount`, `published`
- `lastfm_dataset.csv` tiene: `country`, `genre_tag`, `rank_global`, `rank_by_country`
- El campo en común es `mbid` → hacemos un `left join`

> **CORRECCIÓN #4 — Explosión del merge:** el código original hacía el merge sin deduplicar `lastfm_dataset`,  
> que tiene el mismo `mbid` repetido hasta 15 veces (porque un mismo track aparece en varios endpoints).  
> **IMPACTO:** 34.417 filas se convertían en 512.092 filas (multiplicación por duplicados).  
> **SOLUCIÓN:** deduplicar `lastfm_dataset` por `mbid` antes del merge, priorizando el `country` más informativo  
> (país real > GLOBAL > UNKNOWN).

In [ ]:
# Cargar fuentes (si las tienes ya guardadas, carga desde disco)

df_raw    = pd.read_csv('../../data/raw/backup_tracks.csv')        # generado por track.getInfo

df_lastfm = pd.read_csv('../../data/raw/lastfm_dataset.csv')       # pipeline multi-endpoint

# df_raw.merge(df_tags,left_on='mbid',right_on='name',how='left')

print('backup_tracks shape:', df_raw.shape)
print('lastfm_dataset shape:', df_lastfm.shape)


backup_tracks shape: (34417, 9)
lastfm_dataset shape: (59999, 10)


In [ ]:
df_raw.info()
df_raw.head(3)

In [ ]:
# CORRECCIÓN #4: deduplicar lastfm_dataset por mbid antes del merge
# Prioridad de country: país real (Spain, Mexico...) > GLOBAL > UNKNOWN

df_lastfm_sorted = df_lastfm.copy()
df_lastfm_sorted['_country_priority'] = df_lastfm_sorted['country'].map(
    lambda x: 2 if x == 'UNKNOWN' else (1 if x == 'GLOBAL' else 0)
)
df_lastfm_sorted = df_lastfm_sorted.sort_values('_country_priority')

# Nos quedamos con las columnas que aporta lastfm y no tiene backup
df_country = (
    df_lastfm_sorted
    .drop_duplicates(subset=['mbid'], keep='first')
    [['mbid', 'country', 'genre_tag', 'rank_global', 'rank_by_country']]
)

print(f'df_country (deduplicado): {df_country.shape}')
print(df_country['country'].value_counts().head(10))

In [ ]:
# Merge por mbid — left join para conservar todos los tracks de backup
df_merged = df_raw.merge(df_country, on='mbid', how='left')

print(f'df_merged shape: {df_merged.shape}')  # debe ser igual a df_raw.shape[0]
print(f'Columnas: {df_merged.columns.tolist()}')
df_merged.head(5)

In [ ]:
df_merged.info()

print('\n--- Distribución de country ---')
print(df_merged['country'].value_counts().head(15))

print('\n--- Distribución de genre_tag ---')
print(df_merged['genre_tag'].value_counts().head(15))

In [22]:
# Guardar df_merged para no repetir el proceso
df_merged.to_csv('df_merged-data.csv', index=False)
print('✅ df_merged-data.csv guardado')

NameError: name 'df_merged' is not defined

---
## 6. Limpieza de datos

A partir de aquí trabajamos sobre `df_merged`. Si ya tienes el CSV guardado, puedes cargar desde aquí directamente.

In [ ]:
# Punto de entrada alternativo: cargar df_merged-data.csv directamente
# df_merged = pd.read_csv('df_merged-data.csv')

df_clean = df_merged.copy()

# Tipos numéricos correctos
for col in ['duration', 'listeners', 'playcount']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Strings limpios
for col in ['name', 'artist']:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print('Tipos convertidos:')
print(df_clean[['duration', 'listeners', 'playcount']].dtypes)

In [ ]:
print('Nulos antes de limpiar:')
print(df_clean.isnull().sum())

In [ ]:
# CORRECCIÓN #3: limitar dropna() a columnas críticas solamente
# name y artist son la identidad mínima del track

antes = len(df_clean)
df_clean = df_clean.dropna(subset=['name', 'artist'])
print(f'Filas eliminadas por name/artist nulo: {antes - len(df_clean):,}')
print(f'Filas restantes: {len(df_clean):,}')

In [ ]:
# Limpieza de la columna 'tag'
#
# DUDA RESUELTA: las 47 filas de tipo float son NaN reales.
# Pandas usa float para representar NaN en columnas object.
# Las filas '[]' son strings de listas vacías (tracks sin tags en la API).
#
# Verificación de tipos:
print('Tipos en columna tag:')
print(df_clean['tag'].apply(type).value_counts())

In [ ]:
def clean_tag(x):
    """Extrae el nombre del primer tag válido. Devuelve NaN si no hay tag."""
    if pd.isna(x):
        return np.nan

    # Si es string, convertir a objeto Python con ast.literal_eval
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return np.nan

    # Si es lista no vacía con dicts
    if isinstance(x, list) and len(x) > 0 and isinstance(x[0], dict):
        return x[0].get('name', np.nan)  # primer tag = el más relevante según Last.fm

    return np.nan

df_clean['tag'] = df_clean['tag'].apply(clean_tag)

print(f'Tracks con tag asignado: {df_clean["tag"].notna().sum():,} / {len(df_clean):,}')
print(f'Tags únicos: {df_clean["tag"].nunique()}')
df_clean['tag'].value_counts().head(10)

In [ ]:
# Parsear 'published' como fecha real
# Formato en el CSV: '19 Dec 2008, 19:45'
df_clean['published_date'] = pd.to_datetime(
    df_clean['published'],
    format='%d %b %Y, %H:%M',
    errors='coerce'  # NaT si el formato no coincide
).dt.date

print(f'Fechas válidas: {df_clean["published_date"].notna().sum():,} / {len(df_clean):,}')
df_clean[['published', 'published_date']].head(5)

In [ ]:
# DUDA RESUELTA: country GLOBAL y UNKNOWN
# GLOBAL = tracks de chart.getTopTracks (los más populares del mundo)
# UNKNOWN = tracks de tag.getTopTracks (sin país asignado)
# NO eliminar estas filas del análisis de popularidad — son datos válidos
# Solo excluirlos cuando el análisis geográfico los requiera

print('Distribución de country en df_clean:')
print(df_clean['country'].value_counts(dropna=False))

# Filtrado geográfico solo para análisis por país (NO modifica df_clean)
n_global_unknown = df_clean['country'].isin(['UNKNOWN', 'GLOBAL']).sum()
print(f'\nFilas con country GLOBAL o UNKNOWN: {n_global_unknown:,}')
print('Nota: se usan en análisis de popularidad pero se excluyen del análisis geográfico.')

In [ ]:
print('--- df_clean final ---')
df_clean.info()

print('\n--- Nulos por columna ---')
print(df_clean.isnull().sum())

---
## 7. Feature Engineering

> **CORRECCIÓN #5 — Unidades de `duration`:**  
> `backup_tracks.csv` tiene `duration` en **milisegundos** (viene de `track.getInfo` de Last.fm).  
> `lastfm_dataset.csv` tiene `duration` en **segundos** (viene del pipeline multi-endpoint).  
> Para `df_merged` la columna `duration` viene de `backup_tracks` → es en **milisegundos**.  
> Verificación: The Chain (Fleetwood Mac) = `269000 ms / 60000 = 4.48 min` ✓ (duración real: 4:29)

In [ ]:
# duration_min: duración en minutos
# CORRECCIÓN #5: duration en df_merged está en MILISEGUNDOS → dividir por 60000
# Verificación: 269000 ms / 60000 = 4.48 min (The Chain, Fleetwood Mac) ✓
df_clean['duration_min'] = df_clean['duration'] / 60000

print('Estadísticas de duration_min:')
print(df_clean['duration_min'].describe().round(2))

In [ ]:
# is_short_track: 1 si dura menos de 2.5 min (formato TikTok/Reels)
df_clean['is_short_track'] = (df_clean['duration_min'] < 2.5).astype(int)

print(f'Canciones cortas (<2.5 min): {df_clean["is_short_track"].sum():,} ({df_clean["is_short_track"].mean()*100:.1f}%)')

In [ ]:
# is_hit: 1 si playcount >= percentil 90 (top 10% más populares)
threshold = df_clean['playcount'].quantile(0.90)
df_clean['is_hit'] = (df_clean['playcount'] >= threshold).astype(int)

print(f'Umbral de hit (p90): {threshold:,.0f} reproducciones')
print(f'Hits: {df_clean["is_hit"].sum():,} ({df_clean["is_hit"].mean()*100:.0f}%)')

In [ ]:
# CORRECCIÓN #6 — División por cero:
# Usamos .replace(0, np.nan) para proteger cualquier ratio con denominador

# playcount_per_listener: cuántas veces escucha el track cada oyente (engagement)
df_clean['playcount_per_listener'] = (
    df_clean['playcount'] / df_clean['listeners'].replace(0, np.nan)
)

print('playcount_per_listener:')
print(df_clean['playcount_per_listener'].describe().round(2))

In [ ]:
# Transformaciones logarítmicas
# playcount y listeners tienen distribución muy sesgada → log1p normaliza
df_clean['log_playcount'] = np.log1p(df_clean['playcount'])
df_clean['log_listeners'] = np.log1p(df_clean['listeners'])

print('Skewness:')
print(f'  playcount:     {df_clean["playcount"].skew():.2f}')
print(f'  log_playcount: {df_clean["log_playcount"].skew():.2f}  ← más cercano a 0')

In [ ]:
# Ratios adicionales de popularidad
df_clean['popularity_ratio'] = (
    df_clean['playcount'] / df_clean['playcount'].sum()
)

# CORRECCIÓN #6: .replace(0, np.nan) para evitar inf
df_clean['listener_to_play_ratio'] = (
    df_clean['listeners'] / df_clean['playcount'].replace(0, np.nan)
)

print('Ratios creados.')

In [ ]:
# Estadísticas por artista
# CORRECCIÓN #7: el código original usaba 'df' y 'plays_totales_artista' (nombres incorrectos)
# Se usa df_clean y los nombres correctos

artist_stats = df_clean.groupby('artist').agg(
    artist_track_count     =('name', 'count'),
    artist_total_playcount =('playcount', 'sum')
).reset_index()

df_clean = df_clean.merge(artist_stats, on='artist', how='left')

# Peso del track en el catálogo del artista
# CORRECCIÓN #6: .replace(0, np.nan) para evitar división por cero
df_clean['track_share_of_artist'] = (
    df_clean['playcount'] / df_clean['artist_total_playcount'].replace(0, np.nan)
)

print('Features de artista añadidas.')

In [ ]:
# df_analysis: dataset completo listo para análisis
df_analysis = df_clean.copy()

print(f'✅ df_analysis: {df_analysis.shape[0]:,} filas, {df_analysis.shape[1]} columnas')
print(f'Columnas: {df_analysis.columns.tolist()}')
df_analysis.describe().round(2)

---
## 8. EDA — Análisis Exploratorio

In [ ]:
# 8.1 Distribuciones de playcount (lineal vs log)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

p99 = df_clean['playcount'].quantile(0.99)

axes[0].hist(df_clean['playcount'].clip(upper=p99), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución playcount (lineal)')
axes[0].set_xlabel('Reproducciones')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

axes[1].hist(df_clean['log_playcount'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribución log_playcount ← más normal')
axes[1].set_xlabel('log(1 + playcount)')

plt.suptitle('¿Por qué usamos transformación logarítmica?', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Skewness playcount:     {df_clean["playcount"].skew():.2f}')
print(f'Skewness log_playcount: {df_clean["log_playcount"].skew():.2f}')

In [ ]:
# 8.2 Distribución de duration_min con boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dur_ok   = df_clean['duration_min'].dropna()
dur_clip = dur_ok.clip(upper=dur_ok.quantile(0.99))

axes[0].hist(dur_clip, bins=50, color='seagreen', edgecolor='white')
axes[0].set_title('Distribución duración (minutos)')
axes[0].set_xlabel('Minutos')
axes[0].axvline(2.5, color='red', linestyle='--', label='2.5 min (umbral corta)')
axes[0].legend()

axes[1].boxplot(dur_clip, vert=True, patch_artist=True,
                boxprops=dict(facecolor='seagreen', alpha=0.5))
axes[1].set_title('Boxplot duración')
axes[1].set_ylabel('Minutos')

plt.tight_layout()
plt.show()

print(f'Media duración: {dur_ok.mean():.2f} min')
print(f'Canciones cortas (<2.5 min): {(dur_ok < 2.5).sum():,} ({(dur_ok < 2.5).mean()*100:.1f}%)')

In [ ]:
# 8.3 Top 15 artistas (código original mantenido, ahora con df_clean correcto)
top_artists = (
    df_clean
    .groupby('artist')[['playcount', 'name']]
    .agg({'playcount': 'sum', 'name': 'count'})
    .rename(columns={'playcount': 'total_plays', 'name': 'n_tracks'})
    .sort_values('total_plays', ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_artists.sort_values('total_plays').plot.barh(
    y='total_plays', ax=axes[0], color='steelblue', legend=False
)
axes[0].set_title('Top 15 artistas por reproducciones')
axes[0].set_xlabel('Reproducciones totales')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

top_artists.sort_values('n_tracks').plot.barh(
    y='n_tracks', ax=axes[1], color='coral', legend=False
)
axes[1].set_title('Top 15 artistas por nº de tracks')
axes[1].set_xlabel('Nº de tracks')

plt.tight_layout()
plt.show()

top5_share = top_artists['total_plays'].head(5).sum() / df_clean['playcount'].sum() * 100
print(f'Top 5 artistas concentran {top5_share:.1f}% de reproducciones')

In [ ]:
# 8.4 Análisis por tag (género) — solo tracks con tag asignado
df_con_tag = df_clean.dropna(subset=['tag']).copy()

if len(df_con_tag) > 0:
    stats_tag = (
        df_con_tag
        .groupby('tag')
        .agg(
            n_tracks        =('name', 'count'),
            plays_medio     =('playcount', 'mean'),
            plays_total     =('playcount', 'sum'),
            engagement_medio=('playcount_per_listener', 'mean'),
        )
        .reset_index()
    )
    top10_tag = stats_tag.sort_values('plays_total', ascending=False).head(10)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(top10_tag['tag'], top10_tag['plays_total'] / 1e6,
                 color=sns.color_palette('Blues_r', 10))
    axes[0].invert_yaxis()
    axes[0].set_title('Top 10 tags por reproducciones totales')
    axes[0].set_xlabel('Millones')

    axes[1].barh(top10_tag['tag'], top10_tag['n_tracks'],
                 color=sns.color_palette('Oranges_r', 10))
    axes[1].invert_yaxis()
    axes[1].set_title('Top 10 tags por nº de tracks')
    axes[1].set_xlabel('Nº de tracks')

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No hay tracks con tag para analizar.')

In [ ]:
# 8.5 Análisis geográfico — excluimos GLOBAL y UNKNOWN para este análisis
df_geo = df_clean[~df_clean['country'].isin(['UNKNOWN', 'GLOBAL'])].dropna(subset=['country'])

if len(df_geo) > 0:
    stats_pais = (
        df_geo
        .groupby('country')
        .agg(
            n_tracks       =('name', 'count'),
            plays_medio    =('playcount', 'mean'),
            engagement_medio=('playcount_per_listener', 'mean'),
        )
        .sort_values('plays_medio', ascending=False)
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(stats_pais['country'], stats_pais['plays_medio'] / 1e6,
                 color=sns.color_palette('Blues_r', len(stats_pais)))
    axes[0].invert_yaxis()
    axes[0].set_title('Popularidad media de tracks por país')
    axes[0].set_xlabel('Playcount medio (millones)')

    axes[1].barh(stats_pais['country'], stats_pais['n_tracks'],
                 color=sns.color_palette('Oranges_r', len(stats_pais)))
    axes[1].invert_yaxis()
    axes[1].set_title('Nº de tracks por país')

    plt.suptitle('Análisis geográfico (excluye GLOBAL y UNKNOWN)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('ℹ️  Pocos tracks con país asignado en df_merged. El análisis geográfico es limitado.')
    print('   Para enriquecer: aumentar las páginas en geo.getTopTracks en el pipeline.')

In [ ]:
# 8.6 Distribución de playcount por is_hit e is_short_track
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_clean.boxplot(column='log_playcount', by='is_hit', ax=axes[0])
axes[0].set_xlabel('is_hit (0=No, 1=Sí)')
axes[0].set_ylabel('log_playcount')
plt.sca(axes[0])
plt.title('log_playcount por is_hit')

df_clean.boxplot(column='log_playcount', by='is_short_track', ax=axes[1])
axes[1].set_xlabel('is_short_track (0=Larga, 1=Corta)')
axes[1].set_ylabel('log_playcount')
plt.sca(axes[1])
plt.title('log_playcount por is_short_track')

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 9. Correlaciones

In [ ]:
# Spearman entre duration_min y playcount (código original mantenido)
idx = df_clean[['duration_min', 'playcount']].dropna().index
rho, p = spearmanr(df_clean.loc[idx, 'duration_min'], df_clean.loc[idx, 'playcount'])

print(f'Spearman duration_min vs playcount: ρ={rho:.3f}, p={p:.4f}')
print(f'Interpretación: {"significativo" if p < 0.05 else "no significativo"} (α=0.05)')

In [ ]:
# Heatmap de correlaciones de Spearman
# CORRECCIÓN #8: columnas filtradas a las que realmente existen en df_clean
cols_posibles = [
    'log_playcount', 'log_listeners', 'playcount_per_listener',
    'duration_min', 'is_short_track', 'is_hit',
    'artist_track_count', 'track_share_of_artist', 'popularity_ratio'
]
cols_numericas = [c for c in cols_posibles if c in df_clean.columns]

corr = df_clean[cols_numericas].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlaciones de Spearman', fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

if 'log_playcount' in corr.columns:
    corr_target = corr['log_playcount'].drop('log_playcount').abs().sort_values(ascending=False)
    print('Variables más correlacionadas con log_playcount:')
    for feat, r in corr_target.head(4).items():
        signo = '+' if corr.loc['log_playcount', feat] > 0 else '-'
        print(f'  {signo} {feat}: ρ={r:.3f}')

---
## 10. Tests estadísticos

In [ ]:
# 10.1 Mann-Whitney U: canciones cortas vs largas (código original completado)
# H0: la distribución de playcount es igual entre tracks cortos y largos

short = df_clean[df_clean['is_short_track'] == 1]['playcount'].dropna()
long  = df_clean[df_clean['is_short_track'] == 0]['playcount'].dropna()

stat_mw, p_mw = mannwhitneyu(short, long, alternative='two-sided')

print('=== Mann-Whitney U: cortas vs largas ===')
print(f'  U: {stat_mw:.2f} | p-valor: {p_mw:.6f}')
print(f'  → {"se rechaza H0" if p_mw < 0.05 else "no se rechaza H0"} (α=0.05)')
if p_mw < 0.05:
    mas_popular = 'cortas' if short.median() > long.median() else 'largas'
    print(f'  → Las canciones {mas_popular} tienen mayor playcount mediano.')
    print(f'     Mediana cortas: {short.median():,.0f} | Mediana largas: {long.median():,.0f}')

In [ ]:
# 10.2 Kruskal-Wallis: diferencias por género (tag)
# CORRECCIÓN #9: el original usaba groupby('genre') — la columna se llama 'tag'
# H0: la distribución de playcount es igual en todos los géneros

df_con_tag_kw = df_clean.dropna(subset=['tag', 'playcount'])

# Solo tags con al menos 5 tracks (grupos muy pequeños no son estadísticamente fiables)
tags_validos = df_con_tag_kw['tag'].value_counts()
tags_validos = tags_validos[tags_validos >= 5].index.tolist()

grupos = [
    df_con_tag_kw[df_con_tag_kw['tag'] == tag]['playcount'].values
    for tag in tags_validos
]

if len(grupos) >= 2:
    stat_kw, p_kw = kruskal(*grupos)
    print('=== Kruskal-Wallis: diferencias por tag/género ===')
    print(f'  Géneros con ≥5 tracks: {len(grupos)}')
    print(f'  H: {stat_kw:.2f} | p-valor: {p_kw:.6f}')
    print(f'  → {"se rechaza H0" if p_kw < 0.05 else "no se rechaza H0"} (α=0.05)')
    if p_kw < 0.05:
        print('  → Hay diferencias significativas de popularidad entre géneros.')
else:
    print('⚠️  Insuficientes grupos para Kruskal-Wallis.')

In [ ]:
# 10.3 Análisis visual de duración por rangos (código original completado)
rangos = pd.cut(
    df_clean['duration_min'],
    bins=[0, 1.5, 2.5, 3.5, 4.5, 6, 100],
    labels=['<1.5m', '1.5-2.5m', '2.5-3.5m', '3.5-4.5m', '4.5-6m', '>6m']
)
pop_por_rango = df_clean.groupby(rangos, observed=True)['log_playcount'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pop_por_rango.plot.bar(ax=axes[0], color=sns.color_palette('RdYlGn', 6))
axes[0].set_title('Popularidad media por rango de duración')
axes[0].set_ylabel('log_playcount medio')
axes[0].tick_params(axis='x', rotation=30)

df_clean.boxplot(column='log_playcount', by='is_short_track', ax=axes[1])
axes[1].set_xlabel('is_short_track (0=No, 1=Sí)')
axes[1].set_ylabel('log_playcount')
plt.sca(axes[1])
plt.title('Popularidad: corta vs larga')

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 11. Modelado

### 11.1 Regresión OLS explicativa sobre `log_playcount`

In [ ]:
# Regresión OLS (código original mantenido)
# Variables explicativas: duration_min y playcount_per_listener

df_model = df_clean[['log_playcount', 'duration_min', 'playcount_per_listener']].dropna()

X_ols = df_model[['duration_min', 'playcount_per_listener']]
X_ols = sm.add_constant(X_ols)
y_ols = df_model['log_playcount']

model_ols = sm.OLS(y_ols, X_ols).fit()
print(model_ols.summary())

### 11.2 Random Forest — Clasificación y Regresión

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, r2_score, mean_absolute_error

try:
    from xgboost import XGBClassifier
    XGBOOST = True
    print('✅ XGBoost disponible')
except ImportError:
    XGBOOST = False
    print('⚠️  XGBoost no instalado: pip install xgboost')

In [ ]:
# Preparar features para ML
# Definición explícita de X e y

df_ml = df_clean.copy()
le_tag = LabelEncoder()
df_ml['tag_encoded'] = le_tag.fit_transform(df_ml['tag'].fillna('unknown'))

# FEATURES: columnas que usamos para predecir
FEATURES = [
    'log_listeners',           # oyentes en log (mejor predictor de popularidad)
    'duration_min',            # duración en minutos
    'is_short_track',          # formato corto (<2.5 min)
    'tag_encoded',             # género
    'artist_track_count',      # cuántos tracks tiene el artista
    'track_share_of_artist',   # peso de este track en el catálogo del artista
    'playcount_per_listener',  # engagement
]
FEATURES = [f for f in FEATURES if f in df_ml.columns]

# y para clasificación: is_hit (0 o 1)
# y para regresión:     log_playcount (continuo)
df_ml_ok = df_ml[FEATURES + ['log_playcount', 'is_hit']].dropna()

X = df_ml_ok[FEATURES]
y_clf = df_ml_ok['is_hit']
y_reg = df_ml_ok['log_playcount']

print(f'Dataset ML: {len(df_ml_ok):,} filas | {len(FEATURES)} features')
print(f'Features: {FEATURES}')
print(f'Hits: {y_clf.sum():,} ({y_clf.mean()*100:.0f}%)')

In [ ]:
# Train / Test split — stratify para conservar proporción de hits
X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
_, _, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# Clasificación: ¿es un hit? (is_hit)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_pred_clf = rf_clf.predict(X_test)

print('=== Clasificación (is_hit) ===')
print(classification_report(y_clf_test, y_pred_clf, target_names=['No hit', 'Hit']))

In [ ]:
# Regresión: predecir log_playcount
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred_reg = rf_reg.predict(X_test)

print('=== Regresión (log_playcount) ===')
print(f'R²:  {r2_score(y_reg_test, y_pred_reg):.3f}')
print(f'MAE: {mean_absolute_error(y_reg_test, y_pred_reg):.3f}')

In [ ]:
# Feature importance (código original completado)
importancias = pd.Series(
    rf_clf.feature_importances_, index=FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
importancias.plot.bar(ax=ax, color=sns.color_palette('Blues_r', len(importancias)))
ax.set_title('Feature Importance — Random Forest (is_hit)', fontweight='bold')
ax.set_ylabel('Importancia relativa')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('Las 3 variables más importantes:')
for feat, val in importancias.head(3).items():
    print(f'  → {feat}: {val*100:.1f}%')

In [ ]:
# Función de predicción final (código original completado)
def predecir_hit(nombre, artista, duracion_min, genero, oyentes_estimados):
    """Predice la probabilidad de que una canción sea un hit."""
    genero_enc = le_tag.transform([genero])[0] if genero in le_tag.classes_ else 0

    datos = pd.DataFrame([{
        'log_listeners'          : np.log1p(oyentes_estimados),
        'duration_min'           : duracion_min,
        'is_short_track'         : int(duracion_min < 2.5),
        'tag_encoded'            : genero_enc,
        'artist_track_count'     : 1,
        'track_share_of_artist'  : 1.0,
        'playcount_per_listener' : 5.0,
    }])
    datos = datos[FEATURES]

    prob = rf_clf.predict_proba(datos)[0][1] * 100
    clasificacion = '🚀 Hit potencial' if prob >= 70 else ('🟡 Potencial medio' if prob >= 45 else '📉 Bajo potencial')

    print('=' * 50)
    print(f'  🎵 {nombre} — {artista}')
    print('=' * 50)
    print(f'  Probabilidad de hit: {prob:.1f}%')
    print(f'  Clasificación:       {clasificacion}')
    print(f'  Género:              {genero}')
    print(f'  Duración:            {duracion_min:.1f} min{" (corta ⏱️)" if duracion_min < 2.5 else ""}')
    print('=' * 50)
    return prob

predecir_hit('Mi Canción', 'Mi Artista', 3.2, 'rock', 50000)

---
## 12. Conclusiones

| Análisis | Qué evaluar |
|---|---|
| Distribución playcount | Muy sesgada → necesaria transformación log |
| Mann-Whitney | p < 0.05 → hay diferencia real de popularidad entre cortas y largas |
| Kruskal-Wallis | p < 0.05 → el género influye en la popularidad |
| Spearman | Signo de ρ → si negativo, canciones cortas son más populares |
| Random Forest (clf) | Ver F1 en clase 'Hit'. F1 > 0.7 es aceptable |
| Feature importance | `log_listeners` suele ser el predictor dominante |

### Limitaciones
- `df_merged` hereda las limitaciones de ambas fuentes: tags solo en ~10% de tracks, country con mucha presencia de GLOBAL/UNKNOWN.
- `published` en `backup_tracks` es la fecha de adición al catálogo de Last.fm, no la fecha de lanzamiento original.
- `lastfm_dataset` aporta country y genre_tag, pero la cobertura de mbid compartidos con backup_tracks es parcial.

In [ ]:
# Guardar modelos y encoders
os.makedirs('models', exist_ok=True)

pickle.dump(rf_clf,  open('models/modelo_hits_clf.pkl', 'wb'))
pickle.dump(rf_reg,  open('models/modelo_plays_reg.pkl', 'wb'))
pickle.dump(le_tag,  open('models/le_tag.pkl', 'wb'))

with open('models/features.txt', 'w') as f:
    f.write('\n'.join(FEATURES))

print('✅ Modelos guardados en /models/')
print(f'   Features: {FEATURES}')

---
## Apéndice: Registro de errores detectados y correcciones

| # | Error | Impacto | Corrección |
|---|---|---|---|
| 1 | **API sin validación de `status_code`** — `.json()` sin comprobar si la respuesta fue OK | `JSONDecodeError` o datos incorrectos silenciosos si la API devuelve error | Función `get_json()` centralizada con `status_code == 200` y `try/except` |
| 2 | **Tags solo guardaban `name`** — se perdían `count` y `reach` | Pérdida de información de popularidad del tag | `df_tags` construido con los tres campos |
| 3 | **`dropna()` global destructivo** — eliminaba filas por NaN en cualquier columna | Pérdida masiva de datos en columnas no críticas | `dropna(subset=['mbid'])` y `dropna(subset=['name', 'artist'])` |
| 4 | **Explosión del merge** — merge sin deduplicar `lastfm_dataset` (un mbid aparece hasta 15 veces) | 34.417 filas → 512.092 (multiplicación por duplicados del pipeline) | Deduplicación de `lastfm_dataset` por mbid antes del merge, priorizando country real > GLOBAL > UNKNOWN |
| 5 | **Unidades de `duration`** — `backup_tracks` tiene ms, `lastfm_dataset` tiene segundos | Si se mezclan fuentes, `duration_min` sería incorrecto | Verificación: `269000 ms / 60000 = 4.48 min` ✓. En `df_merged` la columna `duration` viene de `backup_tracks` → dividir por `60000` |
| 6 | **División por cero** — ratios sin `.replace(0, np.nan)` | Genera `inf` en filas con 0 listeners o 0 playcount | `.replace(0, np.nan)` en todos los denominadores |
| 7 | **Variables indefinidas** — celdas 97-101 usaban `df` y `plays_totales_artista` | `NameError` al ejecutar | Unificado en un bloque con `df_clean` y nombres correctos |
| 8 | **`cols_numericas` con columnas inexistentes** — heatmap referenciaba columnas no creadas | `KeyError` al ejecutar | Filtro `[c for c in cols_posibles if c in df_clean.columns]` |
| 9 | **`kruskal()` con `groupby('genre')`** — la columna se llama `tag` | `KeyError: 'genre'` | Corregido a `groupby('tag')` con filtro de grupos ≥ 5 tracks |
| — | **DUDA: 47 filas float en `tag`** | Confusión sobre tipo de dato | Son `NaN` reales. Pandas usa `float` para `NaN` en columnas `object`. Las `[]` son strings de listas vacías (distinto caso). |